# Step 4: Advanced Analysis & Insights

## Objective
Draw meaningful conclusions, identify patterns, and provide actionable recommendations based on the analysis.

## Goals
- Identify revenue drivers and optimization opportunities
- Analyze customer segmentation
- Forecast demand patterns
- Create actionable recommendations
- Generate comprehensive report
- Document key findings

## 4.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import sys
sys.path.append('..')
from src import data_processing, visualization, analysis

sns.set_style('whitegrid')
%matplotlib inline

## 4.2 Load Cleaned Data

In [ ]:
# Load the cleaned dataset
data_path = '../data/processed/hotel_bookings_cleaned.csv'
df = data_processing.load_data(data_path)
print(f"Data loaded: {df.shape}")

## 4.3 Comprehensive Analysis

In [ ]:
# Run complete analysis
results = analysis.analyze_bookings(df)

print("\n=== COMPREHENSIVE ANALYSIS RESULTS ===")
print("\nSummary:")
for key, value in results['summary'].items():
    print(f"  {key}: {value}")

print("\nCancellations:")
for key, value in results['cancellations'].items():
    print(f"  {key}: {value}")

print("\nRevenue:")
for key, value in results['revenue'].items():
    print(f"  {key}: {value}")

## 4.4 Customer Segmentation Analysis

In [ ]:
# Analyze customer segments by multiple dimensions
print("\n=== CUSTOMER SEGMENTS ===")

# By market segment
if 'market_segment' in df.columns:
    print("\nBy Market Segment:")
    market_segments = analysis.customer_segmentation(df, 'market_segment')
    print(market_segments)

# By customer type
if 'customer_type' in df.columns:
    print("\nBy Customer Type:")
    customer_types = analysis.customer_segmentation(df, 'customer_type')
    print(customer_types)

# By hotel type
if 'hotel' in df.columns:
    print("\nBy Hotel Type:")
    hotel_types = analysis.customer_segmentation(df, 'hotel')
    print(hotel_types)

## 4.5 Revenue Driver Analysis

In [ ]:
# Analyze factors affecting revenue
print("\n=== REVENUE DRIVERS ===")

# Revenue by hotel type
if 'hotel' in df.columns and 'adr' in df.columns:
    revenue_by_hotel = df.groupby('hotel').agg({
        'adr': ['mean', 'median', 'count'],
        'is_canceled': 'mean' if 'is_canceled' in df.columns else None
    }).round(2)
    print("\nRevenue by Hotel Type:")
    print(revenue_by_hotel)

# Revenue by market segment
if 'market_segment' in df.columns and 'adr' in df.columns:
    revenue_by_segment = df.groupby('market_segment').agg({
        'adr': ['mean', 'count']
    }).round(2)
    print("\nRevenue by Market Segment:")
    print(revenue_by_segment)

## 4.6 Cancellation Root Cause Analysis

In [ ]:
# Analyze cancellation patterns
if 'is_canceled' in df.columns:
    print("\n=== CANCELLATION PATTERNS ===")
    
    # Cancellation by lead time
    if 'lead_time' in df.columns:
        lead_time_bins = [0, 7, 14, 30, 90, 365]
        df['lead_time_category'] = pd.cut(df['lead_time'], bins=lead_time_bins)
        cancel_by_lead = df.groupby('lead_time_category')['is_canceled'].agg(['sum', 'count', 'mean'])
        cancel_by_lead.columns = ['Cancellations', 'Total', 'Cancel_Rate']
        print("\nCancellations by Lead Time:")
        print(cancel_by_lead)
    
    # Cancellation by customer type
    if 'customer_type' in df.columns:
        cancel_by_customer = df.groupby('customer_type')['is_canceled'].mean()
        print("\nCancellation Rate by Customer Type:")
        print((cancel_by_customer * 100).round(2))

## 4.7 Demand Forecasting Preparation

In [ ]:
# Prepare data for demand forecasting
if 'arrival_date_month' in df.columns:
    print("\n=== DEMAND PATTERNS ===")
    
    monthly_demand = df.groupby('arrival_date_month').agg({
        'booking_id': 'count',
        'adr': 'mean',
        'is_canceled': 'mean' if 'is_canceled' in df.columns else None
    }).round(2)
    monthly_demand.columns = ['Bookings', 'Avg_Price', 'Cancel_Rate']
    
    print("\nMonthly Demand Summary:")
    print(monthly_demand)
    
    # Identify peak and low seasons
    peak_month = monthly_demand['Bookings'].idxmax()
    low_month = monthly_demand['Bookings'].idxmin()
    print(f"\nPeak Season: {peak_month} ({monthly_demand.loc[peak_month, 'Bookings']:.0f} bookings)")
    print(f"Low Season: {low_month} ({monthly_demand.loc[low_month, 'Bookings']:.0f} bookings)")

## 4.8 Key Performance Indicators (KPIs)

In [ ]:
# Calculate key performance indicators
print("\n=== KEY PERFORMANCE INDICATORS ===")

total_bookings = len(df)
completed_bookings = len(df[df['is_canceled'] == 0]) if 'is_canceled' in df.columns else len(df)
cancellation_rate = (total_bookings - completed_bookings) / total_bookings if 'is_canceled' in df.columns else 0

print(f"\nTotal Bookings: {total_bookings:,}")
print(f"Completed Bookings: {completed_bookings:,}")
print(f"Cancellation Rate: {cancellation_rate*100:.2f}%")

if 'adr' in df.columns:
    avg_adr = df['adr'].mean()
    print(f"Average Daily Rate: ${avg_adr:.2f}")

if 'adults' in df.columns:
    avg_guests = df['adults'].mean()
    print(f"Average Guests per Booking: {avg_guests:.1f}")

if 'lead_time' in df.columns:
    avg_lead = df['lead_time'].mean()
    print(f"Average Lead Time: {avg_lead:.1f} days")

## 4.9 Actionable Recommendations

In [ ]:
print("\n=== ACTIONABLE RECOMMENDATIONS ===")

recommendations = []

# Based on cancellation rate
if 'is_canceled' in df.columns:
    cancel_rate = df['is_canceled'].mean() * 100
    if cancel_rate > 30:
        recommendations.append("Implement stricter cancellation policies to reduce high cancellation rate")
    elif cancel_rate > 20:
        recommendations.append("Consider offering flexible booking options to reduce cancellations")

# Based on seasonality
if 'arrival_date_month' in df.columns:
    recommendations.append("Implement dynamic pricing strategy based on seasonal demand patterns")
    recommendations.append("Prepare marketing campaigns for low-season to boost bookings")

# Based on lead time
if 'lead_time' in df.columns:
    avg_lead = df['lead_time'].mean()
    if avg_lead < 30:
        recommendations.append("Focus on short-term booking channels and promotions")
    else:
        recommendations.append("Develop early-bird booking incentives")

# General recommendations
recommendations.append("Focus marketing efforts on high-value customer segments")
recommendations.append("Optimize room inventory management for peak seasons")
recommendations.append("Enhance customer experience to reduce cancellations")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

## 4.10 Executive Summary

### Key Findings
1. [Finding 1]
2. [Finding 2]
3. [Finding 3]
4. [Finding 4]
5. [Finding 5]

### Critical Insights
- **Booking Volume**: [Insight]
- **Revenue Impact**: [Insight]
- **Cancellation Risk**: [Insight]
- **Seasonal Patterns**: [Insight]
- **Customer Behavior**: [Insight]

### Strategic Recommendations
1. [Recommendation 1]
2. [Recommendation 2]
3. [Recommendation 3]
4. [Recommendation 4]
5. [Recommendation 5]

### Next Steps
- Implement recommended strategies
- Monitor KPIs regularly
- Conduct monthly/quarterly reviews
- Update analysis with new data
- Share insights with stakeholders